## Stock Direction Predictor — LSTM Classifier
Predicts next-day UP/DOWN direction for RELIANCE.NS using an LSTM trained on 10 years of daily data.

Setting random seeds first to ensure reproducible results across runs.

In [29]:
import os
os.environ['PYTHONHASHSEED'] = '100'
import random
random.seed(100)
import numpy as np
np.random.seed(100)
import tensorflow as tf
tf.random.set_seed(100)

In [30]:
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.callbacks import EarlyStopping, ModelCheckpoint

### Data Collection
Downloading 10 years of daily OHLCV data for RELIANCE.NS from Yahoo Finance via yfinance.

In [31]:
ticker="RELIANCE.NS"
df=yf.download(ticker,period="10y",interval="1d")
df.columns=[col[0] for col in df.columns]
df=df[['Open','High','Low','Close','Volume']]

print(f"Data downloaded: {len(df)} rows")
print(df.tail())

[*********************100%***********************]  1 of 1 completed

Data downloaded: 2471 rows
                   Open         High          Low        Close    Volume
Date                                                                    
2026-06-18  1330.000000  1333.900024  1322.000000  1328.099976  15494549
2026-06-19  1328.000000  1338.199951  1305.300049  1309.500000  24887034
2026-06-22  1316.699951  1344.900024  1314.099976  1326.500000  12931213
2026-06-23  1328.900024  1333.000000  1304.000000  1309.500000  15400184
2026-06-24  1305.699951  1322.000000  1297.500000  1313.599976  11028739


### Feature Engineering
Computing technical indicators: Moving Averages (MA7, MA21), Volatility, Volume Change, RSI, MACD, and Bollinger Band Width. MACD_Signal, BB_Upper, BB_Lower are intermediate calculations used to derive the final features.

In [32]:
df['MA7'] = df['Close'].rolling(7).mean()
df['MA21'] = df['Close'].rolling(21).mean()
df['Volatility'] = df['Close'].rolling(7).std()
df['Volume_Change'] = df['Volume'].pct_change()

delta = df['Close'].diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)
avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()
rs = avg_gain / avg_loss
df['RSI'] = 100 - (100 / (1 + rs))

ema12 = df['Close'].ewm(span=12, adjust=False).mean()
ema26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = ema12 - ema26
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']

bb_middle = df['Close'].rolling(20).mean()
bb_std = df['Close'].rolling(20).std()
df['BB_Upper'] = bb_middle + (bb_std * 2)
df['BB_Lower'] = bb_middle - (bb_std * 2)
df['BB_Width'] = (df['BB_Upper'] - df['BB_Lower']) / bb_middle

print(f"Features added. Total columns: {len(df.columns)}")
print(df[['RSI', 'MACD', 'MACD_Hist', 'BB_Width']].tail())

Features added. Total columns: 16
                  RSI       MACD  MACD_Hist  BB_Width
Date                                                 
2026-06-18  54.426640 -11.181864   6.795342  0.095687
2026-06-19  48.651139 -10.302653   6.139642  0.092058
2026-06-22  55.109666  -8.140278   6.641614  0.085512
2026-06-23  50.612726  -7.709468   5.657939  0.080035
2026-06-24  54.276510  -6.957018   5.128311  0.075413


### Label Creation and Scaling
Direction = 1 (UP) if next day's return is positive, else 0 (DOWN). Inf values dropped from zero-volume days. Features scaled using MinMaxScaler.

In [33]:
df['Return'] = df['Close'].pct_change()
df = df.dropna()
df = df.replace([np.inf, -np.inf], np.nan).dropna()

df['Direction'] = (df['Return'].shift(-1) > 0).astype(int)
df = df.dropna()

feature_cols = ['Return', 'MA7', 'MA21', 'Volatility', 'Volume_Change',
                 'RSI', 'MACD', 'MACD_Hist', 'BB_Width']
data = df[feature_cols].values
target = df['Direction'].values

scaler = MinMaxScaler(feature_range=(0,1))
scaled = scaler.fit_transform(data)

print(f"Class balance — UP: {(target==1).sum()}, DOWN: {(target==0).sum()}")

Class balance — UP: 1263, DOWN: 1184


### Train/Test Split
80/20 chronological split — no shuffling to avoid data leakage. Train on past, test on future.

In [34]:
train_size=int(len(scaled)*0.8)
train_data=scaled[:train_size]
test_data=scaled[train_size:]

train_target = target[:train_size]
test_target  = target[train_size:]

### Sequence Builder
Building 60-day rolling windows as input to the LSTM. Each sample is 60 consecutive days of features predicting the direction on day 61.

In [35]:
def create_sequences(dataset,target_arr, window=60):
    X, y = [], []
    for i in range(window, len(dataset)):
        X.append(dataset[i-window:i, :])
        y.append(target_arr[i])
    return np.array(X), np.array(y)
X_train, y_train = create_sequences(train_data, train_target)
X_test, y_test   = create_sequences(test_data, test_target)

In [36]:
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

X_train: (1897, 60, 9), y_train: (1897,)
X_test:  (430, 60, 9),  y_test:  (430,)


### Model Architecture
Single LSTM layer kept intentionally simple — deeper architectures (128→64 stacked) were tested and performed worse on this dataset.

In [37]:
model = Sequential([
    LSTM(units=32, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_2 (LSTM)               (None, 32)                5376      
                                                                 
 dropout_2 (Dropout)         (None, 32)                0         
                                                                 
 dense_4 (Dense)             (None, 16)                528       
                                                                 
 dense_5 (Dense)             (None, 1)                 17        
                                                                 
Total params: 5,921
Trainable params: 5,921
Non-trainable params: 0
_________________________________________________________________


### Class Weights
Without class weighting the model collapsed to predicting BUY for every sample. Balanced weights fix this.

In [38]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(class_weight_dict)

{0: 1.0469094922737308, 1: 0.9571140262361252}


### Training
EarlyStopping monitors val_accuracy with patience=20. Best weights saved automatically via ModelCheckpoint.

In [39]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=20,
    restore_best_weights=True,
    mode='max'
)
checkpoint = ModelCheckpoint(
    'best_lstm_classifier.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop, checkpoint],
    class_weight=class_weight_dict,
    verbose=1
)
print(f"\nTraining complete — stopped at epoch {len(history.history['loss'])}")
print(f"Best val_accuracy: {max(history.history['val_accuracy']):.6f}")

Epoch 1/100
58/60 [============================>.] - ETA: 0s - loss: 0.6957 - accuracy: 0.4865
Epoch 1: val_accuracy improved from -inf to 0.48837, saving model to best_lstm_classifier.h5
60/60 [==============================] - 5s 36ms/step - loss: 0.6959 - accuracy: 0.4834 - val_loss: 0.6943 - val_accuracy: 0.4884
Epoch 2/100
60/60 [==============================] - ETA: 0s - loss: 0.6955 - accuracy: 0.5092
Epoch 2: val_accuracy improved from 0.48837 to 0.52791, saving model to best_lstm_classifier.h5
60/60 [==============================] - 1s 22ms/step - loss: 0.6955 - accuracy: 0.5092 - val_loss: 0.6932 - val_accuracy: 0.5279
Epoch 3/100
57/60 [===========================>..] - ETA: 0s - loss: 0.6942 - accuracy: 0.5038
Epoch 3: val_accuracy did not improve from 0.52791
60/60 [==============================] - 1s 18ms/step - loss: 0.6942 - accuracy: 0.5034 - val_loss: 0.6929 - val_accuracy: 0.5070
Epoch 4/100
60/60 [==============================] - ETA: 0s - loss: 0.6941 - accurac

### Evaluation
Checking overall directional accuracy, per-signal accuracy, and signal distribution.

In [40]:
predicted_proba = model.predict(X_test).flatten()
predicted_direction = (predicted_proba > 0.5).astype(int)

accuracy = (predicted_direction == y_test).mean() * 100
print(f"Test Accuracy (Directional): {accuracy:.2f}%")
print(f"(50% = random guessing)")

signals = np.where(predicted_direction == 1, 'BUY', 'SELL')

results_df = pd.DataFrame({
    'Actual_Direction': np.where(y_test==1, 'UP', 'DOWN'),
    'Predicted_Probability': predicted_proba,
    'Signal': signals,
    'Correct': predicted_direction == y_test
})

print("\nLast 15 predictions:")
print(results_df.tail(15))

print("\nSignal counts:")
print(results_df['Signal'].value_counts())

print(f"\nAccuracy when model says BUY: {results_df[results_df['Signal']=='BUY']['Correct'].mean()*100:.2f}%")
print(f"Accuracy when model says SELL: {results_df[results_df['Signal']=='SELL']['Correct'].mean()*100:.2f}%")

14/14 [==============================] - 1s 7ms/step
Test Accuracy (Directional): 53.02%
(50% = random guessing)

Last 15 predictions:
    Actual_Direction  Predicted_Probability Signal  Correct
415             DOWN               0.496997   SELL     True
416             DOWN               0.497749   SELL     True
417               UP               0.497845   SELL    False
418             DOWN               0.498409   SELL     True
419               UP               0.497790   SELL    False
420               UP               0.497473   SELL    False
421               UP               0.496747   SELL    False
422               UP               0.496155   SELL    False
423               UP               0.496260   SELL    False
424             DOWN               0.496407   SELL     True
425             DOWN               0.496046   SELL     True
426               UP               0.496906   SELL    False
427             DOWN               0.496776   SELL     True
428               UP     

### Probability Distribution Check
Verifying the model isn't collapsing to a near-constant prediction. A healthy std dev means it's genuinely differentiating between samples.

In [41]:
print(f"Predicted probability — min: {predicted_proba.min():.6f}, max: {predicted_proba.max():.6f}")
print(f"Predicted probability — std dev: {predicted_proba.std():.6f}")

Predicted probability — min: 0.489796, max: 0.509769
Predicted probability — std dev: 0.003641
